# SINASC 2023 — Região Sudeste: dados para avaliação independente de viabilidade causal

**Projeto:** Início precoce do pré-natal e baixo peso ao nascer no Brasil.

Este notebook prepara dados reais do SINASC 2023 para que a professora avalie, de forma independente, se existe um desenho de inferência causal defensável. Ele apenas **lê, filtra, documenta e exporta** os dados. Não aplica critérios de elegibilidade, não constrói exposição nem desfecho e não ajusta nenhum modelo (escore de propensão, AIPW, *matching* ou outro).

| Arquivo gerado em `outputs/compartilhamento_professora/` | Conteúdo |
| --- | --- |
| `sinasc_sudeste_2023.parquet` | Todos os nascimentos de 2023 de mães residentes em MG, ES, RJ ou SP |
| `sinasc_sudeste_2023_amostra.csv` | 100.000 registros sorteados do arquivo acima: amostra aleatória simples, sem reposição, semente 42 |

**Execução:** use a `.venv` do projeto (`projetos/big-data-sinasc-causal-ml/.venv`). A entrada é `data/processed/sinasc/sinasc_2023.parquet`.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import duckdb
import numpy as np
import pandas as pd


def localizar_raiz(inicio: Path) -> Path:
    # Sobe a árvore de pastas até a raiz do projeto (pasta com src/ e data/processed/).
    for pasta in (inicio, *inicio.parents):
        if (pasta / "src").is_dir() and (pasta / "data" / "processed").is_dir():
            return pasta
    raise FileNotFoundError(f"Raiz do projeto não encontrada a partir de {inicio}")


RAIZ = localizar_raiz(Path.cwd().resolve())
PARQUET_2023 = RAIZ / "data" / "processed" / "sinasc" / "sinasc_2023.parquet"
MANIFESTO_2023 = RAIZ / "outputs" / "temporal" / "manifestos" / "sinasc_2023.json"
SAIDA = RAIZ / "outputs" / "compartilhamento_professora"
SAIDA.mkdir(parents=True, exist_ok=True)
ARQ_PARQUET = SAIDA / "sinasc_sudeste_2023.parquet"
ARQ_CSV = SAIDA / "sinasc_sudeste_2023_amostra.csv"

UFS_SUDESTE = {"31": "MG", "32": "ES", "33": "RJ", "35": "SP"}
LISTA_UFS_SQL = ", ".join(f"'{uf}'" for uf in UFS_SUDESTE)
SEMENTE = 42
N_AMOSTRA = 100_000

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
VALIDACOES: list[dict] = []


def registrar(verificacao: str, ok: bool, observado, esperado="", status_falha: str = "FALHA") -> None:
    VALIDACOES.append({"verificacao": verificacao, "status": "OK" if ok else status_falha,
                       "observado": str(observado), "esperado": str(esperado)})


def sha256_arquivo(caminho: Path, bloco: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(caminho, "rb") as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b""):
            h.update(parte)
    return h.hexdigest()


print("Raiz do projeto:", RAIZ)
print("DuckDB", duckdb.__version__, "| pandas", pd.__version__, "| NumPy", np.__version__)

Raiz do projeto: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml
DuckDB 1.5.5 | pandas 3.0.6 | NumPy 2.4.6


## 1. Fonte e granularidade

- **Fonte:** Sistema de Informações sobre Nascidos Vivos (SINASC), Ministério da Saúde, Portal de Dados Abertos do SUS. O arquivo oficial de 2023 (CSV compactado) foi convertido para Parquet pelo projeto, sem alteração de conteúdo: todos os campos permanecem como texto.
- **Unidade de observação:** 1 linha = 1 nascido vivo = 1 Declaração de Nascido Vivo (DNV). Gêmeos geram uma linha por recém-nascido.
- Não há identificador longitudinal da mãe. Todos os campos são registrados no momento do nascimento.

A célula abaixo confere a proveniência do Parquet (SHA-256 registrado no manifesto do projeto), a unicidade do registro e o ano das datas de nascimento. As datas são lidas apenas para essa conferência e não são exportadas.

In [2]:
manifesto = json.loads(MANIFESTO_2023.read_text(encoding="utf-8"))
sha_local = sha256_arquivo(PARQUET_2023)
sha_esperado = manifesto["conversao"]["parquet_sha256"]
registrar("Parquet 2023 idêntico ao registrado no manifesto do projeto (SHA-256)", sha_local == sha_esperado,
          sha_local[:16], sha_esperado[:16])

con.execute(f"CREATE OR REPLACE VIEW sinasc_2023 AS SELECT * FROM read_parquet('{PARQUET_2023.as_posix()}', file_row_number = true)")
fonte = con.execute('''
    SELECT count(*) AS registros_brasil,
           count(DISTINCT contador) AS contadores_distintos,
           count(*) FILTER (WHERE try_strptime(DTNASC, '%d%m%Y') IS NULL
                               OR year(try_strptime(DTNASC, '%d%m%Y')) <> 2023) AS datas_invalidas_ou_fora_de_2023
    FROM sinasc_2023
''').df()
f = fonte.iloc[0]
registrar("Fonte: uma linha por DNV (CONTADOR único)", f.registros_brasil == f.contadores_distintos,
          f"{f.registros_brasil} linhas / {f.contadores_distintos} contadores", "iguais")
registrar("Fonte: todas as datas de nascimento válidas e em 2023", f.datas_invalidas_ou_fora_de_2023 == 0,
          f.datas_invalidas_ou_fora_de_2023, 0)
registrar("Fonte: quantidade de registros igual à do manifesto", f.registros_brasil == manifesto["conversao"]["n_registros"],
          f.registros_brasil, manifesto["conversao"]["n_registros"])
pd.Series({
    "Página oficial": manifesto["pagina_oficial"],
    "Arquivo oficial (URL)": manifesto["url"],
    "Data de acesso": manifesto["data_acesso"],
    "SHA-256 do ZIP oficial": manifesto["sha256"],
    "SHA-256 do Parquet local": sha_local,
    "Registros no Brasil (2023)": int(f.registros_brasil),
    "Datas inválidas ou fora de 2023": int(f.datas_invalidas_ou_fora_de_2023),
}, name="valor").to_frame()

,valor
Página oficial,https://dadosabertos.saude.gov.br/dataset/sistema-de-informacao-sobre-nascidos-vivos-sinasc
Arquivo oficial (URL),https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2023_csv.zip
Data de acesso,2026-09-29
SHA-256 do ZIP oficial,c4ce57f7aa539023e915b6820a466047466f87d48c81f8c1714ae21c21be510e
SHA-256 do Parquet local,66ad723138fced4de6206ca204efd3a5297662cb69a877e4bb9332baac662347
Registros no Brasil (2023),2537576
Datas inválidas ou fora de 2023,0


## 2. Filtro regional: município de residência da mãe

O recorte usa **somente** `CODMUNRES` (código IBGE de 6 dígitos do município de residência da mãe). Os dois primeiros dígitos identificam a UF: 31 (MG), 32 (ES), 33 (RJ) e 35 (SP). O local do parto (`CODMUNNASC`) e o estabelecimento (`CODESTAB`) **não** são usados.

Nenhum registro é removido por valores ausentes, mês de início do pré-natal, tipo de gravidez ou peso.

In [3]:
COLUNAS_ORIGINAIS = [
    "IDADEMAE", "ESCMAE2010", "RACACORMAE", "ESTCIVMAE",
    "PARIDADE", "QTDGESTANT", "QTDPARTNOR", "QTDPARTCES", "QTDFILVIVO", "QTDFILMORT",
    "MESPRENAT", "CONSPRENAT", "CONSULTAS", "KOTELCHUCK",
    "GRAVIDEZ", "SEMAGESTAC", "GESTACAO", "TPMETESTIM",
    "PARTO", "PESO", "SEXO", "IDANOMAL",
]
COLUNAS_EXPORTACAO = ["ID_REGISTRO", "CODMUNRES", "UF_RESIDENCIA_COD", "UF_RESIDENCIA_SIGLA", *COLUNAS_ORIGINAIS]

con.execute(f'''
    CREATE OR REPLACE TABLE sudeste AS
    SELECT
        row_number() OVER (ORDER BY file_row_number) AS ID_REGISTRO,
        CODMUNRES,
        substr(CODMUNRES, 1, 2) AS UF_RESIDENCIA_COD,
        CASE substr(CODMUNRES, 1, 2) WHEN '31' THEN 'MG' WHEN '32' THEN 'ES'
                                     WHEN '33' THEN 'RJ' WHEN '35' THEN 'SP' END AS UF_RESIDENCIA_SIGLA,
        {", ".join(COLUNAS_ORIGINAIS)}
    FROM sinasc_2023
    WHERE substr(CODMUNRES, 1, 2) IN ({LISTA_UFS_SQL})
    ORDER BY ID_REGISTRO
''')

por_uf = con.execute('''
    SELECT UF_RESIDENCIA_COD AS uf_cod, UF_RESIDENCIA_SIGLA AS uf, count(*) AS registros,
           count(DISTINCT CODMUNRES) AS municipios,
           count(*) FILTER (WHERE CODMUNRES LIKE '__0000') AS municipio_ignorado_UF0000
    FROM sudeste GROUP BY ALL ORDER BY uf_cod
''').df()
fonte_uf = con.execute(f'''
    SELECT substr(CODMUNRES, 1, 2) AS uf_cod, count(*) AS registros_na_fonte
    FROM sinasc_2023 WHERE substr(CODMUNRES, 1, 2) IN ({LISTA_UFS_SQL}) GROUP BY 1
''').df()
por_uf = por_uf.merge(fonte_uf, on="uf_cod")
por_uf["percentual"] = (100 * por_uf.registros / por_uf.registros.sum()).round(2)
N_SUDESTE = int(por_uf.registros.sum())
print(f"Registros do Sudeste em 2023: {N_SUDESTE:,}".replace(",", "."))
por_uf

Registros do Sudeste em 2023: 966.160


,uf_cod,uf,registros,municipios,municipio_ignorado_UF0000,registros_na_fonte,percentual
0,31,MG,233918,854,2,233918,24.21
1,32,ES,52187,78,0,52187,5.40
2,33,RJ,176145,93,8,176145,18.23
3,35,SP,503910,645,7,503910,52.16


## 3. Dicionário das variáveis exportadas

Os códigos originais do SINASC são mantidos **como texto**, exatamente como publicados (por exemplo, `MESPRENAT = '02'` e `QTDFILMORT = '00'`). As únicas colunas criadas são `ID_REGISTRO` e as duas colunas de UF.

O "papel possível" é uma sugestão para orientar a discussão, e não uma decisão. Na proposta atual do projeto:

- **T** (exposição) é o início do pré-natal no 1º–3º mês (T = 1) versus 4º–6º mês (T = 0), a partir de `MESPRENAT`;
- **Y** (desfecho) é o baixo peso ao nascer, `PESO` < 2.500 g;
- a população restringe-se a gestação única com peso entre 500 e 6.000 g;
- as covariáveis X são idade, escolaridade, raça/cor, situação conjugal, paridade, perdas fetais e UF.

Essas regras **não** foram aplicadas aos arquivos, que contêm todos os registros.

In [4]:
DICIONARIO = pd.DataFrame([
    ("ID_REGISTRO", "Identificador pseudônimo, sequencial na ordem do arquivo oficial dentro do recorte. Liga CSV e Parquet. Não é o CONTADOR do SINASC", "inteiro", "chave técnica", "—"),
    ("CODMUNRES", "Município de residência da mãe (código IBGE, 6 dígitos)", "UF0000 = município ignorado na UF", "filtro do recorte; confundidor geográfico; agrupamento", "declarado no parto; a residência pode ter mudado na gestação"),
    ("UF_RESIDENCIA_COD", "UF de residência (2 primeiros dígitos de CODMUNRES)", "31, 32, 33, 35", "X candidata", "idem"),
    ("UF_RESIDENCIA_SIGLA", "Sigla da UF de residência", "MG, ES, RJ, SP", "descritiva", "idem"),
    ("IDADEMAE", "Idade da mãe (anos)", "número de anos; 99 provável ignorado", "X candidata (confundidor)", "registrada no parto; praticamente igual à do início da gestação"),
    ("ESCMAE2010", "Escolaridade da mãe", "0 sem escolaridade; 1 fundamental I; 2 fundamental II; 3 médio; 4 superior incompleto; 5 superior completo; 9 ignorado", "X candidata (confundidor socioeconômico)", "declarada no parto"),
    ("RACACORMAE", "Raça/cor da mãe", "1 branca; 2 preta; 3 amarela; 4 parda; 5 indígena", "X candidata (confundidor)", "característica fixa"),
    ("ESTCIVMAE", "Situação conjugal da mãe", "1 solteira; 2 casada; 3 viúva; 4 separada/divorciada; 5 união estável; 9 ignorada", "X candidata (apoio social)", "registrada no parto; pode ter mudado durante a gestação"),
    ("PARIDADE", "Primeira gestação ou não (derivado pelo sistema)", "0 nulípara; 1 multípara", "X candidata (histórico gestacional)", "anterior à gestação atual"),
    ("QTDGESTANT", "Número de gestações anteriores", "contagem; 99 provável ignorado", "histórico gestacional (candidata à discussão)", "anterior à gestação atual"),
    ("QTDPARTNOR", "Número de partos vaginais anteriores", "contagem; 99 provável ignorado", "histórico gestacional (candidata à discussão)", "anterior à gestação atual"),
    ("QTDPARTCES", "Número de partos cesáreos anteriores", "contagem; 99 provável ignorado", "histórico gestacional (candidata à discussão)", "anterior à gestação atual"),
    ("QTDFILVIVO", "Número de filhos vivos", "contagem; 99 provável ignorado", "histórico gestacional (candidata à discussão)", "anterior à gestação atual"),
    ("QTDFILMORT", "Número de perdas fetais e abortos anteriores", "contagem; 99 provável ignorado", "X candidata (histórico obstétrico)", "anterior à gestação; baixa concordância relatada na validação da DNV"),
    ("MESPRENAT", "Mês de gestação em que o pré-natal foi iniciado", "01 a 09; 99 ignorado", "base da exposição T", "registrado retrospectivamente no parto; erro de memória ou registro possível"),
    ("CONSPRENAT", "Número de consultas de pré-natal", "contagem; 99 provável ignorado", "mediadora potencial (não ajustar como confundidor)", "acumula-se depois do início do pré-natal"),
    ("CONSULTAS", "Faixa do número de consultas", "1 nenhuma; 2 de 1 a 3; 3 de 4 a 6; 4 sete ou mais; 9 ignorado", "mediadora potencial", "pós-exposição"),
    ("KOTELCHUCK", "Índice de Kotelchuck de adequação do pré-natal", "1 a 5 e 9. Significado não confirmado no layout; convenção usual: 1 não fez pré-natal, 2 inadequado, 3 intermediário, 4 adequado, 5 mais que adequado, 9 não classificado", "auditoria (combina início e consultas)", "parcialmente função da exposição"),
    ("GRAVIDEZ", "Tipo de gravidez", "1 única; 2 dupla; 3 tripla ou mais; 9 ignorado", "elegibilidade", "conhecido no parto"),
    ("SEMAGESTAC", "Semanas de gestação ao nascer", "número de semanas", "mediadora ou collider; elegibilidade temporal", "fim da gestação; limita a oportunidade de início tardio"),
    ("GESTACAO", "Faixa de idade gestacional ao nascer", "1 menos de 22 sem; 2 22–27; 3 28–31; 4 32–36; 5 37–41; 6 42 ou mais; 9 ignorado", "mediadora ou collider", "fim da gestação"),
    ("TPMETESTIM", "Método de estimativa da idade gestacional", "1 exame físico; 2 outro método; 9 ignorado; o código 8 aparece nos dados, mas não consta do layout disponível", "qualidade da idade gestacional", "registro do parto"),
    ("PARTO", "Tipo de parto", "1 vaginal; 2 cesáreo; 9 ignorado", "pós-exposição (auditoria)", "no parto"),
    ("PESO", "Peso ao nascer (gramas)", "gramas", "base do desfecho Y; qualidade (faixa plausível)", "medido no nascimento"),
    ("SEXO", "Sexo do recém-nascido", "1 masculino; 2 feminino; 0 ignorado", "preditor do peso; possível modificador", "definido na concepção; não causado pelo pré-natal"),
    ("IDANOMAL", "Anomalia congênita identificada", "1 sim; 2 não; 9 ignorado", "auditoria (causa de baixo peso; detecção pode depender do pré-natal)", "registrada no nascimento"),
], columns=["coluna", "descricao", "codificacao", "papel_possivel", "temporalidade_observacao"])

registrar("Dicionário cobre exatamente as colunas exportadas", DICIONARIO.coluna.tolist() == COLUNAS_EXPORTACAO,
          len(DICIONARIO), len(COLUNAS_EXPORTACAO))
DICIONARIO.style.hide(axis="index").set_properties(**{"text-align": "left"})

coluna,descricao,codificacao,papel_possivel,temporalidade_observacao
ID_REGISTRO,"Identificador pseudônimo, sequencial na ordem do arquivo oficial dentro do recorte. Liga CSV e Parquet. Não é o CONTADOR do SINASC",inteiro,chave técnica,—
CODMUNRES,"Município de residência da mãe (código IBGE, 6 dígitos)",UF0000 = município ignorado na UF,filtro do recorte; confundidor geográfico; agrupamento,declarado no parto; a residência pode ter mudado na gestação
UF_RESIDENCIA_COD,UF de residência (2 primeiros dígitos de CODMUNRES),"31, 32, 33, 35",X candidata,idem
UF_RESIDENCIA_SIGLA,Sigla da UF de residência,"MG, ES, RJ, SP",descritiva,idem
IDADEMAE,Idade da mãe (anos),número de anos; 99 provável ignorado,X candidata (confundidor),registrada no parto; praticamente igual à do início da gestação
ESCMAE2010,Escolaridade da mãe,0 sem escolaridade; 1 fundamental I; 2 fundamental II; 3 médio; 4 superior incompleto; 5 superior completo; 9 ignorado,X candidata (confundidor socioeconômico),declarada no parto
RACACORMAE,Raça/cor da mãe,1 branca; 2 preta; 3 amarela; 4 parda; 5 indígena,X candidata (confundidor),característica fixa
ESTCIVMAE,Situação conjugal da mãe,1 solteira; 2 casada; 3 viúva; 4 separada/divorciada; 5 união estável; 9 ignorada,X candidata (apoio social),registrada no parto; pode ter mudado durante a gestação
PARIDADE,Primeira gestação ou não (derivado pelo sistema),0 nulípara; 1 multípara,X candidata (histórico gestacional),anterior à gestação atual
QTDGESTANT,Número de gestações anteriores,contagem; 99 provável ignorado,histórico gestacional (candidata à discussão),anterior à gestação atual


**Campos deliberadamente não exportados:**

- identificadores e datas: `CONTADOR`, `DTNASC`, `HORANASC`, `DTNASCMAE`, `DTULTMENST`, `DTCADASTRO`, `DTRECEBIM`, `DTRECORIGA`, `DTDECLARAC`;
- local de ocorrência e estabelecimento: `CODESTAB`, `CODMUNNASC`, `LOCNASC`;
- ocupação e naturalidade da mãe: `CODOCUPMAE`, `NATURALMAE`, `CODMUNNATU`, `CODUFNATU`;
- idade do pai: `IDADEPAI`;
- dados de lote e sistema;
- Apgar e demais campos da assistência ao parto.

A ocupação da mãe (`CODOCUPMAE`) poderia ser um indicador socioeconômico. Ela foi omitida por ser muito detalhada e, combinada com o município, aumentar o risco de reidentificação. Pode ser incluída se a professora considerar necessário.

**Risco de reidentificação:** em municípios pequenos, a combinação de `CODMUNRES` com idade, raça/cor e paridade pode tornar um registro único. Os arquivos destinam-se apenas à avaliação acadêmica e não devem ser republicados.

## 4. Valores ausentes e códigos de ignorado

- **nulo:** campo vazio no arquivo oficial, que aparece como célula vazia no CSV.
- **código de ignorado:** 9 nas variáveis categóricas, 99 nas contagens, na idade e no mês de início, e 0 em `SEXO`.

Nenhum valor é imputado ou recodificado.

In [5]:
CODIGOS_IGNORADOS = {
    "IDADEMAE": ["99"], "ESCMAE2010": ["9"], "RACACORMAE": ["9"], "ESTCIVMAE": ["9"], "PARIDADE": ["9", "99"],
    "QTDGESTANT": ["99"], "QTDPARTNOR": ["99"], "QTDPARTCES": ["99"], "QTDFILVIVO": ["99"], "QTDFILMORT": ["99"],
    "MESPRENAT": ["99"], "CONSPRENAT": ["99"], "CONSULTAS": ["9"], "KOTELCHUCK": ["9"], "GRAVIDEZ": ["9"],
    "SEMAGESTAC": [], "GESTACAO": ["9"], "TPMETESTIM": ["9"], "PARTO": ["9"], "PESO": [], "SEXO": ["0"],
    "IDANOMAL": ["9"], "CODMUNRES": [],
}
linhas = []
for coluna, codigos in CODIGOS_IGNORADOS.items():
    cond = " OR ".join(f"{coluna} = '{c}'" for c in codigos) or "false"
    nulos, ignorados = con.execute(f"SELECT count(*) FILTER (WHERE {coluna} IS NULL), count(*) FILTER (WHERE {cond}) FROM sudeste").fetchone()
    linhas.append({"coluna": coluna, "codigo_ignorado": ", ".join(codigos) or "—", "nulos": nulos,
                   "pct_nulos": round(100 * nulos / N_SUDESTE, 3), "codigo_ignorado_n": ignorados,
                   "pct_codigo_ignorado": round(100 * ignorados / N_SUDESTE, 3)})
ausentes = pd.DataFrame(linhas)
ausentes

,coluna,codigo_ignorado,nulos,pct_nulos,codigo_ignorado_n,pct_codigo_ignorado
0,IDADEMAE,99,0,0.000,11,0.001
1,ESCMAE2010,9,2666,0.276,1524,0.158
2,RACACORMAE,9,9027,0.934,0,0.000
3,ESTCIVMAE,9,2445,0.253,1563,0.162
4,PARIDADE,"9, 99",0,0.000,0,0.000
5,QTDGESTANT,99,4041,0.418,33,0.003
6,QTDPARTNOR,99,6415,0.664,53,0.005
7,QTDPARTCES,99,6153,0.637,61,0.006
8,QTDFILVIVO,99,4889,0.506,39,0.004
9,QTDFILMORT,99,7185,0.744,72,0.007


In [6]:
# Faixas de MESPRENAT e de PESO sem excluir nada: úteis para a professora dimensionar os critérios de elegibilidade.
con.execute('''
    SELECT 'MESPRENAT' AS variavel,
           CASE WHEN MESPRENAT IS NULL THEN 'nulo' WHEN MESPRENAT = '99' THEN '99 ignorado'
                WHEN try_cast(MESPRENAT AS INTEGER) BETWEEN 1 AND 3 THEN '1–3'
                WHEN try_cast(MESPRENAT AS INTEGER) BETWEEN 4 AND 6 THEN '4–6'
                WHEN try_cast(MESPRENAT AS INTEGER) BETWEEN 7 AND 9 THEN '7–9' ELSE 'outro' END AS faixa,
           count(*) AS registros
    FROM sudeste GROUP BY ALL
    UNION ALL
    SELECT 'PESO',
           CASE WHEN try_cast(PESO AS INTEGER) IS NULL THEN 'nulo ou não numérico'
                WHEN try_cast(PESO AS INTEGER) < 500 THEN '< 500 g'
                WHEN try_cast(PESO AS INTEGER) < 2500 THEN '500–2.499 g'
                WHEN try_cast(PESO AS INTEGER) <= 6000 THEN '2.500–6.000 g' ELSE '> 6.000 g' END,
           count(*)
    FROM sudeste GROUP BY ALL
    UNION ALL
    SELECT 'GRAVIDEZ', coalesce(GRAVIDEZ, 'nulo'), count(*) FROM sudeste GROUP BY ALL
    ORDER BY 1, 2
''').df().assign(percentual=lambda d: (100 * d.registros / N_SUDESTE).round(3))

,variavel,faixa,registros,percentual
0,GRAVIDEZ,1,941105,97.407
1,GRAVIDEZ,2,24287,2.514
2,GRAVIDEZ,3,466,0.048
3,GRAVIDEZ,9,27,0.003
4,GRAVIDEZ,nulo,275,0.028
5,MESPRENAT,1–3,830297,85.938
6,MESPRENAT,4–6,97929,10.136
7,MESPRENAT,7–9,15080,1.561
8,MESPRENAT,99 ignorado,14054,1.455
9,MESPRENAT,nulo,8800,0.911


## 5. Amostra aleatória de 100.000 registros

- **Universo:** todos os registros do Sudeste, sem nenhum filtro.
- **Método:** amostragem aleatória simples, **sem reposição**, de 100.000 valores de `ID_REGISTRO`, com `numpy.random.default_rng(42).choice(...)` aplicado à lista ordenada de identificadores.
- **Independência da exposição e do desfecho:** o sorteio não usa nenhuma variável, nem exposição, nem desfecho, nem UF. As proporções naturais são preservadas dentro da variação aleatória esperada, como mostra a comparação abaixo (valor z com correção para população finita; |z| > 4 seria alerta).

In [7]:
def sortear_ids(ids_ordenados: np.ndarray, n: int, semente: int) -> np.ndarray:
    rng = np.random.default_rng(semente)
    return np.sort(rng.choice(ids_ordenados, size=n, replace=False))


ids = con.execute("SELECT ID_REGISTRO FROM sudeste ORDER BY ID_REGISTRO").fetchnumpy()["ID_REGISTRO"]
ids_amostra = sortear_ids(ids, N_AMOSTRA, SEMENTE)
SHA_IDS = hashlib.sha256("\n".join(map(str, ids_amostra)).encode()).hexdigest()
registrar("Amostragem reprodutível: segundo sorteio com semente 42 idêntico ao primeiro",
          np.array_equal(ids_amostra, sortear_ids(ids, N_AMOSTRA, SEMENTE)), SHA_IDS[:16], "mesma lista")
registrar("Amostra sem reposição: 100.000 identificadores distintos",
          len(np.unique(ids_amostra)) == N_AMOSTRA, len(np.unique(ids_amostra)), N_AMOSTRA)
con.register("ids_amostra", pd.DataFrame({"ID_REGISTRO": ids_amostra}))
print("SHA-256 da lista de ID_REGISTRO sorteados:", SHA_IDS)


def comparar(expressao: str, rotulo: str) -> pd.DataFrame:
    return con.execute(f'''
        SELECT '{rotulo}' AS variavel, coalesce(CAST({expressao} AS VARCHAR), 'nulo') AS categoria,
               count(*) AS n_sudeste, count(a.ID_REGISTRO) AS n_amostra
        FROM sudeste s LEFT JOIN ids_amostra a USING (ID_REGISTRO) GROUP BY ALL''').df()


comparacao = pd.concat([
    comparar("UF_RESIDENCIA_SIGLA", "UF"),
    comparar("CASE WHEN try_cast(MESPRENAT AS INTEGER) BETWEEN 1 AND 3 THEN '1–3' WHEN try_cast(MESPRENAT AS INTEGER) BETWEEN 4 AND 6 THEN '4–6' WHEN try_cast(MESPRENAT AS INTEGER) BETWEEN 7 AND 9 THEN '7–9' ELSE 'desconhecido' END", "MESPRENAT (faixa)"),
    comparar("CASE WHEN try_cast(PESO AS INTEGER) < 2500 THEN '< 2.500 g' WHEN try_cast(PESO AS INTEGER) >= 2500 THEN '≥ 2.500 g' END", "PESO (faixa)"),
    comparar("GRAVIDEZ", "GRAVIDEZ"),
], ignore_index=True)
p, q = comparacao.n_sudeste / N_SUDESTE, comparacao.n_amostra / N_AMOSTRA
erro_padrao = np.sqrt(p * (1 - p) / N_AMOSTRA * (N_SUDESTE - N_AMOSTRA) / (N_SUDESTE - 1))
comparacao["pct_sudeste"] = (100 * p).round(3)
comparacao["pct_amostra"] = (100 * q).round(3)
comparacao["z"] = ((q - p) / erro_padrao).round(2)
registrar("Amostra preserva as proporções naturais (|z| ≤ 4)", comparacao.z.abs().max() <= 4,
          f"max |z| = {comparacao.z.abs().max():.2f}", "≤ 4", status_falha="ALERTA")
comparacao.sort_values(["variavel", "categoria"]).reset_index(drop=True)

SHA-256 da lista de ID_REGISTRO sorteados: 26872ff14e751a01554b62f605a1e852c7f6c62c59ea6a49388adbc6934f720b


,variavel,categoria,n_sudeste,n_amostra,pct_sudeste,pct_amostra,z
0,GRAVIDEZ,1,941105,97456,97.407,97.456,1.04
1,GRAVIDEZ,2,24287,2477,2.514,2.477,-0.78
2,GRAVIDEZ,3,466,47,0.048,0.047,-0.19
3,GRAVIDEZ,9,27,3,0.003,0.003,0.13
4,GRAVIDEZ,nulo,275,17,0.028,0.017,-2.27
5,MESPRENAT (faixa),1–3,830297,85887,85.938,85.887,-0.49
6,MESPRENAT (faixa),4–6,97929,10183,10.136,10.183,0.52
7,MESPRENAT (faixa),7–9,15080,1520,1.561,1.520,-1.10
8,MESPRENAT (faixa),desconhecido,22854,2410,2.365,2.410,0.98
9,PESO (faixa),< 2.500 g,98009,9965,10.144,9.965,-1.98


## 6. Exportação

- **Parquet:** compressão ZSTD; todos os registros do Sudeste, ordenados por `ID_REGISTRO`.
- **CSV:** UTF-8, separador `;` (o mesmo do arquivo oficial do SINASC), com cabeçalho. Valores ausentes aparecem como campo vazio.

Os códigos originais são texto. Ao abrir o CSV diretamente no Excel, os zeros à esquerda (`'02'`) podem ser removidos. Para evitar isso, use *Dados → De Texto/CSV* e defina as colunas de código como Texto. No pandas, use `pd.read_csv(..., sep=";", dtype=str)`.

In [8]:
LISTA_SQL = ", ".join(COLUNAS_EXPORTACAO)
con.execute(f"COPY (SELECT {LISTA_SQL} FROM sudeste ORDER BY ID_REGISTRO) TO '{ARQ_PARQUET.as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
con.execute(f'''COPY (SELECT {LISTA_SQL} FROM sudeste SEMI JOIN ids_amostra USING (ID_REGISTRO) ORDER BY ID_REGISTRO)
                TO '{ARQ_CSV.as_posix()}' (FORMAT csv, HEADER true, DELIMITER ';')''')
print("Gravados:", ARQ_PARQUET.relative_to(RAIZ), "e", ARQ_CSV.relative_to(RAIZ))

Gravados: outputs\compartilhamento_professora\sinasc_sudeste_2023.parquet e outputs\compartilhamento_professora\sinasc_sudeste_2023_amostra.csv


## 7. Validações básicas

As verificações releem os arquivos gravados em disco e cobrem:

- quantidade de registros;
- distribuição por UF;
- colunas e tipos;
- ausência de identificadores;
- fidelidade dos valores ao arquivo de origem;
- reprodutibilidade e consistência da amostra;
- leitura com pandas e DuckDB.

In [9]:
TIPOS = {c: ("BIGINT" if c == "ID_REGISTRO" else "VARCHAR") for c in COLUNAS_EXPORTACAO}
TIPOS_CSV = "{" + ", ".join(f"'{c}': '{t}'" for c, t in TIPOS.items()) + "}"
con.execute(f"CREATE OR REPLACE VIEW arq_parquet AS SELECT * FROM read_parquet('{ARQ_PARQUET.as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW arq_csv AS SELECT * FROM read_csv('{ARQ_CSV.as_posix()}', delim = ';', header = true, columns = {TIPOS_CSV})")

# Quantidade e UF
n_parquet = con.execute("SELECT count(*) FROM arq_parquet").fetchone()[0]
registrar("Parquet: quantidade igual à da fonte filtrada por CODMUNRES", n_parquet == N_SUDESTE, n_parquet, N_SUDESTE)
uf_arq = con.execute("SELECT UF_RESIDENCIA_COD, count(*) FROM arq_parquet GROUP BY 1 ORDER BY 1").fetchall()
uf_fonte = [(r.uf_cod, r.registros_na_fonte) for r in por_uf.itertuples()]
registrar("Parquet: somente UFs 31, 32, 33 e 35, com contagens idênticas à fonte", uf_arq == uf_fonte, dict(uf_arq), dict(uf_fonte))
registrar("Parquet: UF derivada coerente com CODMUNRES",
          con.execute("SELECT count(*) FROM arq_parquet WHERE substr(CODMUNRES, 1, 2) <> UF_RESIDENCIA_COD").fetchone()[0] == 0, 0, 0)
registrar("Parquet: ID_REGISTRO único", con.execute("SELECT count(*) - count(DISTINCT ID_REGISTRO) FROM arq_parquet").fetchone()[0] == 0, 0, 0)

# Fidelidade aos códigos originais da fonte (mesmos registros e valores, sem nenhuma remoção)
originais = ", ".join(["CODMUNRES", *COLUNAS_ORIGINAIS])
fonte_sudeste = f"SELECT {originais} FROM sinasc_2023 WHERE substr(CODMUNRES, 1, 2) IN ({LISTA_UFS_SQL})"
dif = [con.execute(f"SELECT count(*) FROM ({a} EXCEPT ALL {b})").fetchone()[0] for a, b in
       [(f"SELECT {originais} FROM arq_parquet", fonte_sudeste), (fonte_sudeste, f"SELECT {originais} FROM arq_parquet")]]
registrar("Parquet: códigos originais idênticos aos da fonte (EXCEPT ALL nos dois sentidos)", dif == [0, 0], dif, [0, 0])

# Colunas, tipos e identificadores
PROIBIDAS = {"CONTADOR", "DTNASC", "HORANASC", "DTNASCMAE", "DTULTMENST", "DTCADASTRO", "DTRECEBIM", "DTRECORIGA",
             "DTDECLARAC", "CODESTAB", "CODMUNNASC", "LOCNASC", "CODOCUPMAE", "NATURALMAE", "CODMUNNATU", "CODUFNATU",
             "IDADEPAI", "NUMEROLOTE"}
for nome in ["parquet", "csv"]:
    esquema = con.execute(f"DESCRIBE arq_{nome}").df()
    registrar(f"{nome.upper()}: colunas e tipos exportados conforme o dicionário",
              esquema.column_name.tolist() == COLUNAS_EXPORTACAO and dict(zip(esquema.column_name, esquema.column_type)) == TIPOS,
              f"{len(esquema)} colunas", f"{len(COLUNAS_EXPORTACAO)} colunas")
    proibidas = sorted({c.upper() for c in esquema.column_name} & PROIBIDAS)
    registrar(f"{nome.upper()}: sem identificadores individuais nem datas", not proibidas, proibidas or "nenhum", "nenhum")

# Amostra
n_csv = con.execute("SELECT count(*) FROM arq_csv").fetchone()[0]
registrar("CSV: 100.000 registros distintos", n_csv == N_AMOSTRA and
          con.execute("SELECT count(DISTINCT ID_REGISTRO) FROM arq_csv").fetchone()[0] == N_AMOSTRA, n_csv, N_AMOSTRA)
ids_csv = con.execute("SELECT ID_REGISTRO FROM arq_csv ORDER BY ID_REGISTRO").fetchnumpy()["ID_REGISTRO"]
registrar("CSV: identificadores iguais aos sorteados com semente 42", np.array_equal(ids_csv, ids_amostra),
          hashlib.sha256("\n".join(map(str, ids_csv)).encode()).hexdigest()[:16], SHA_IDS[:16])
dif_amostra = con.execute("SELECT count(*) FROM (SELECT * FROM arq_csv EXCEPT ALL SELECT p.* FROM arq_parquet p SEMI JOIN arq_csv c USING (ID_REGISTRO))").fetchone()[0]
registrar("CSV: valores idênticos aos dos mesmos registros no Parquet", dif_amostra == 0, dif_amostra, 0)

# Leitura com pandas e DuckDB
parquet_pd = pd.read_parquet(ARQ_PARQUET)
csv_pd = pd.read_csv(ARQ_CSV, sep=";", dtype=str, keep_default_na=False, na_values=[""])
registrar("Leitura com pandas e DuckDB: contagens e colunas corretas",
          (len(parquet_pd), len(csv_pd), n_parquet, n_csv) == (N_SUDESTE, N_AMOSTRA, N_SUDESTE, N_AMOSTRA)
          and parquet_pd.columns.tolist() == csv_pd.columns.tolist() == COLUNAS_EXPORTACAO,
          (len(parquet_pd), len(csv_pd), n_parquet, n_csv), (N_SUDESTE, N_AMOSTRA, N_SUDESTE, N_AMOSTRA))
csv_pd["ID_REGISTRO"] = csv_pd["ID_REGISTRO"].astype("int64")
subconjunto = parquet_pd.set_index("ID_REGISTRO").loc[csv_pd.ID_REGISTRO].reset_index().astype({c: "string" for c in COLUNAS_EXPORTACAO[1:]})
registrar("Leitura com pandas: CSV igual ao subconjunto do Parquet",
          subconjunto.equals(csv_pd.astype({c: "string" for c in COLUNAS_EXPORTACAO[1:]})), "iguais", "iguais")
del parquet_pd, subconjunto

validacoes = pd.DataFrame(VALIDACOES)
print("Status:", validacoes.status.value_counts().to_dict())
validacoes

Status: {'OK': 22}


,verificacao,status,observado,esperado
0,Parquet 2023 idêntico ao registrado no manifesto do projeto (SHA-256),OK,66ad723138fced4d,66ad723138fced4d
1,Fonte: uma linha por DNV (CONTADOR único),OK,2537576 linhas / 2537576 contadores,iguais
2,Fonte: todas as datas de nascimento válidas e em 2023,OK,0,0
3,Fonte: quantidade de registros igual à do manifesto,OK,2537576,2537576
4,Dicionário cobre exatamente as colunas exportadas,OK,26,26
5,Amostragem reprodutível: segundo sorteio com semente 42 idêntico ao primeiro,OK,26872ff14e751a01,mesma lista
6,Amostra sem reposição: 100.000 identificadores distintos,OK,100000,100000
7,Amostra preserva as proporções naturais (|z| ≤ 4),OK,max |z| = 2.27,≤ 4
8,Parquet: quantidade igual à da fonte filtrada por CODMUNRES,OK,966160,966160
9,"Parquet: somente UFs 31, 32, 33 e 35, com contagens idênticas à fonte",OK,"{'31': 233918, '32': 52187, '33': 176145, '35': 503910}","{'31': 233918, '32': 52187, '33': 176145, '35': 503910}"


In [10]:
arquivos = pd.DataFrame([
    {"arquivo": str(p.relative_to(RAIZ)), "registros": n, "colunas": len(COLUNAS_EXPORTACAO),
     "tamanho_MB": round(p.stat().st_size / 1024 ** 2, 2), "sha256": sha256_arquivo(p)}
    for p, n in [(ARQ_PARQUET, n_parquet), (ARQ_CSV, n_csv)]
])
assert (validacoes.status != "FALHA").all(), validacoes.query("status == 'FALHA'").to_string()
print("Todas as validações sem falha.")
arquivos

Todas as validações sem falha.


,arquivo,registros,colunas,tamanho_MB,sha256
0,outputs\compartilhamento_professora\sinasc_sudeste_2023.parquet,966160,26,7.84,3c23594538f0d0b4c03e15173bf242f52c8f4d5d6e496484a609270e1d1e513b
1,outputs\compartilhamento_professora\sinasc_sudeste_2023_amostra.csv,100000,26,7.23,d042295094c8212943589262ad6ca3eb20b4f54e2ce82619095a636df97cd5ee
